# RAM-H1200 Bone Segmentation - Kaggle

This notebook assumes the RAM-H1200 dataset is attached as a Kaggle Input or can be downloaded from Hugging Face when Internet is enabled.


In [ ]:
from pathlib import Path
import os

KAGGLE_INPUT = Path('/kaggle/input')
WORKING = Path('/kaggle/working')
PROJECT_DIR = WORKING / 'Thesis' / 'project'
OUTPUT_DIR = WORKING / 'ThesisOutputs'
SAM_CHECKPOINT = WORKING / 'sam_vit_b_01ec64.pth'

CLASSIFIER_OUTPUT = OUTPUT_DIR / 'classifier'
PSEUDO_OUTPUT = OUTPUT_DIR / 'pseudo_masks'
SEG_OUTPUT = OUTPUT_DIR / 'segmentation'


In [ ]:
# Clone your repo if it is not already present in /kaggle/working.
REPO_URL = 'https://github.com/<your-user>/<your-repo>.git'
if not PROJECT_DIR.exists():
    !git clone --branch TN_exp {REPO_URL} {WORKING / 'Thesis'}
%cd {PROJECT_DIR}
!pip install -r requirements.txt


In [ ]:
def find_ram_root(base: Path):
    for candidate in [base, *base.glob('*'), *base.glob('*/*')]:
        if (candidate / 'Segmentation').exists():
            return candidate
        if (candidate / 'train').exists() and (candidate / '_annotations_bone_rle.coco.json').exists():
            return candidate.parent
    return None

RAM_ROOT = find_ram_root(KAGGLE_INPUT)
if RAM_ROOT is None:
    !pip install -q huggingface_hub
    from huggingface_hub import snapshot_download
    RAM_ROOT = Path(snapshot_download(
        repo_id='TokyoTechMagicYang/RAM-H1200-v1',
        repo_type='dataset',
        local_dir=str(WORKING / 'RAM-H1200-v1'),
        local_dir_use_symlinks=False,
    ))

print('RAM-H1200 root:', RAM_ROOT)


In [ ]:
if not SAM_CHECKPOINT.exists():
    !wget -q -O {SAM_CHECKPOINT} https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth
print('SAM checkpoint:', SAM_CHECKPOINT)


## Stage 1 - Train RAM-H1200 hand classifier


In [ ]:
!python train_classifier.py \
  --ram-root "{RAM_ROOT}" \
  --target-columns hand \
  --image-size 384 \
  --batch-size 4 \
  --epochs 25 \
  --output-dir "{CLASSIFIER_OUTPUT}"


## Stage 2 - Pseudo mask generation


In [ ]:
RUN_FULL_PSEUDO_MASKS = False
PROCESS_MODE = '--process-all' if RUN_FULL_PSEUDO_MASKS else '--max-images 10'

!python generate_pseudo_masks.py \
  --ram-root "{RAM_ROOT}" \
  --split val \
  --classifier-checkpoint "{CLASSIFIER_OUTPUT / 'best_classifier.pt'}" \
  --sam-checkpoint "{SAM_CHECKPOINT}" \
  --target-columns hand \
  --image-size 384 \
  --batch-size 1 \
  --num-workers 2 \
  --selection-method bone_hybrid \
  --morphology-fusion-mode components \
  --sam-prompt-mode box_point \
  --save-visuals-limit 10 \
  {PROCESS_MODE} \
  --output-dir "{PSEUDO_OUTPUT}"


## Evaluate pseudo masks against RAM-H1200 GT


In [ ]:
!python evaluate_ramh1200_masks.py \
  --ram-root "{RAM_ROOT}" \
  --split val \
  --pred-mask-root "{PSEUDO_OUTPUT / 'masks'}" \
  --image-size 384


## Supervised U-Net baseline


In [ ]:
!python train_segmentation.py \
  --ram-root "{RAM_ROOT}" \
  --train-split train \
  --val-split val \
  --image-size 384 \
  --batch-size 4 \
  --epochs 25 \
  --output-dir "{SEG_OUTPUT}"
